In [1]:
# Importing the Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler,LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor 

# load the dataset
data = pd.read_excel('BBDM Project for 2nd research article.xlsx')

# Columns to exclude
columns_to_exclude = ['Chainage','Formation','RMC', ]

# Preprocessing: Exclude specified columns
X = data.drop(['PRnet'] + columns_to_exclude, axis=1)
Y = data['PRnet']

# Label encoding for multiple columns
label_encoder = LabelEncoder()
for col in ['Lithology', 'Weathering', 'Rock Strength']:
    X[col] = label_encoder.fit_transform(X[col])


# Train Test Split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)  # Removed 'stratify' as it is not used in regression

# Data Standardization
scaler = RobustScaler()
scaler.fit(X_train)  # Fit on the training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Create a RandomForestRegressor
rf_regressor = RandomForestRegressor(random_state=42)

# Define the parameter grid for Grid Search
param_grid = {
    'n_estimators': list(range(10, 301, 10)),
    'max_depth': list(range(5, 51, 5)),
    'max_features': ['sqrt', 'log2'],
    'min_samples_leaf': list(range(1, 11)),
    'min_samples_split': list(range(2, 11)),
}

# Create GridSearchCV object
grid_search = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, cv=5,
                           scoring='neg_mean_absolute_error', 
                           n_jobs=-1, verbose=2)

# Fit the model to the training data
grid_search.fit(X_train_scaled, Y_train)

# Get the best parameters
best_params = grid_search.best_params_
print("Best Hyperparameters:", best_params)

# Prediction and Model Evaluation
Y_pred = grid_search.predict(X_test_scaled)

# Calculate evaluation metrics
mae = mean_absolute_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("Mean Absolute Error:", mae)
print("R^2 Score:", r2)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Hyperparameters: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Mean Absolute Error: 2.361937317434762
R^2 Score: 0.9371702760354775
